# Automation and workflowing

A rule says what to do with which messages. `automate` looks through the inbox and
acts once per message; run it from cron, a timer, or an Ansible play.

In [ ]:
from amoorion import Amoorion
from amoorion.models import Message

amo = Amoorion()
amo.store.keep([
    Message(platform="hook:generic", text="deploy finished", id="1"),
    Message(platform="hook:generic", text="backup finished", id="2"),
])
rule = {
    "name": "announce-deploys",
    "platform": "hook:generic",
    "contains": "deploy",
    "steps": [{"method": "publish", "params": {"text": "Status: $text", "to": ["dryrun"]}}],
}
for done in amo.automate(rules=[rule]):
    print(done["rule"], done["message_id"], done["ok"], done["steps"][0]["result"][0]["message"]["text"])

Run again, it finds nothing new:

In [ ]:
amo.automate(rules=[rule])

A step that fails leaves the message unhandled, so the next run tries again:

In [ ]:
amo.store.keep([Message(platform="hook:generic", text="deploy two", id="3")])
broken = rule | {"name": "to-nowhere", "contains": "two", "steps": [{"method": "publish", "params": {"text": "$text", "to": ["nowhere"]}}]}
print([done["ok"] for done in amo.automate(rules=[broken])])
print([done["ok"] for done in amo.automate(rules=[broken])])

Rules are plain data, so they can live in YAML next to `amo.save()`:

In [ ]:
import yaml

print(yaml.safe_dump({"rules": [rule]}, sort_keys=False))